## Imports

In [1]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from paths import PROCESSED_DIR

## Load the data file

In [2]:
FILE_NAME = 'final_features.csv'
FILE_PATH = PROCESSED_DIR / FILE_NAME

if not FILE_PATH.exists():
    raise FileNotFoundError('Could not find final_features.csv')

df = pd.read_csv(FILE_PATH)
print('Loaded:', FILE_PATH)
print('Shape:', df.shape)
df.head()

Loaded: ..\resources\data\processed\final_features.csv
Shape: (126530, 8)


,user_id,product_id,up_total_bought,user_total_orders,user_avg_days_between,prod_total_purchases,prod_reorder_ratio,reordered
0,102,4920,3,6,19.542373,816,0.670343,0
1,102,8174,1,6,19.542373,436,0.653670,0
2,102,9839,3,6,19.542373,430,0.746512,0
3,102,12845,1,6,19.542373,88,0.352273,0
4,102,13629,1,6,19.542373,208,0.783654,0


## Train Test split by user

Define features and labels, then split the data into training and test sets by user. Group shuffle on user is used to avoid data leakage.

In [17]:
target = 'reordered'
group = 'user_id'
to_drop = [target, 'user_id', 'product_id']
features = [c for c in df.columns if c not in to_drop]
X = df[features].copy()
y = df[target].copy()
groups = df[group].copy()
print('Feature columns:', features)

Feature columns: ['up_total_bought', 'user_total_orders', 'user_avg_days_between', 'prod_total_purchases', 'prod_reorder_ratio']


In [18]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

In [7]:
print('Train shape:', X_train.shape)
print('test shape:', X_test.shape)
print('Train users:', groups.iloc[train_idx].nunique())
print('test users:', groups.iloc[test_idx].nunique())
print('Overlap in users:', len(set(train_df['user_id']).intersection(set(test_df['user_id']))))

Train shape: (101507, 5)
test shape: (25023, 5)
Train users: 1600
test users: 400
Overlap in users: 0


## Save the split data set

In [21]:
train_df.to_parquet(PROCESSED_DIR / 'train.parquet') 
test_df.to_parquet(PROCESSED_DIR / 'test.parquet')
X.to_parquet(PROCESSED_DIR / 'X.parquet')
y.to_frame().to_parquet(PROCESSED_DIR / 'y.parquet')
groups.to_frame().to_parquet(PROCESSED_DIR / 'groups.parquet')
X_train.to_parquet(PROCESSED_DIR / 'X_train.parquet')
y_train.to_frame().to_parquet(PROCESSED_DIR / 'y_train.parquet')
X_test.to_parquet(PROCESSED_DIR / 'X_test.parquet')
y_test.to_frame().to_parquet(PROCESSED_DIR / 'y_test.parquet')